# Batch Job Title Mapping for Top Companies

Run LLM-based job title to O*NET mapping on top 10 companies by unique title count.

In [2]:
import pandas as pd
import subprocess
import json
import re
import time
import os
from collections import defaultdict
from pathlib import Path

In [ ]:
# Input paths (relative)
titles_path = '../../../data/processed/llm_title_accuracy_test/step-1-extract-unique-titles/company_unique_titles.csv'
taxonomy_path = '../../../data/raw/onet_job_occupation_taxonomy.csv'

# Output directory
output_dir = '../../../data/processed/llm_title_accuracy_test/step-2-llm-onet-mapping/top10_onet_tagged'
os.makedirs(output_dir, exist_ok=True)

print(f'Titles: {titles_path}')
print(f'Taxonomy: {taxonomy_path}')
print(f'Output dir: {output_dir}')

In [4]:
# Load data
df_titles = pd.read_csv(titles_path)
df_taxonomy = pd.read_csv(taxonomy_path)

# O*NET categories
onet_titles = df_taxonomy['Title'].tolist()

# Get top 10 companies
top_companies = df_titles.head(10)

print(f'Total companies: {len(df_titles):,}')
print(f'O*NET categories: {len(onet_titles):,}')
print(f'\nTop 10 companies to process:')
print(top_companies[['company_name', 'total_postings', 'unique_titles_count']])

Total companies: 24,428
O*NET categories: 923

Top 10 companies to process:
                                     company_name  total_postings  \
0                                 The Job Network            1003   
1                                      TEKsystems             529   
2                                            Dice             415   
3                                  Insight Global             418   
4                                          Macy's             333   
5                                  VolunteerMatch             322   
6                                    Apex Systems             325   
7                                          Amazon             343   
8  Liberty Healthcare and Rehabilitation Services            1108   
9                       Maxim Healthcare Staffing             278   

   unique_titles_count  
0                  724  
1                  395  
2                  385  
3                  359  
4                  306  
5             

In [5]:
def map_titles_for_company(company_row, onet_titles, batch_size=30):
    """
    Map job titles to O*NET categories for a single company.
    
    Args:
        company_row: Row from company_unique_titles.csv
        onet_titles: List of O*NET category titles
        batch_size: Number of titles to process per batch
    
    Returns:
        DataFrame with mapped results
    """
    company_name = company_row['company_name']
    titles_list = company_row['unique_titles'].split(';')
    
    print(f'\nProcessing: {company_name}')
    print(f'  Unique titles: {len(titles_list)}')
    
    all_mappings = []
    total = len(titles_list)
    
    for i in range(0, total, batch_size):
        batch = titles_list[i:i+batch_size]
        print(f'  Batch {i//batch_size + 1}/{(total-1)//batch_size + 1}: Processing {i+1}-{min(i+batch_size, total)} / {total}')
        
        prompt = f"""Map each job title to the most relevant O*NET category.

Job titles to map:
{json.dumps(batch)}

Available O*NET categories:
{json.dumps(onet_titles)}

Return JSON only, no explanation:
{{"mappings": [{{"raw": "original title", "onet": "O*NET category"}}]}}
"""
        
        with open('temp_prompt.txt', 'w', encoding='utf-8') as f:
            f.write(prompt)
        
        result = subprocess.run(
            'type temp_prompt.txt | claude --print -',
            capture_output=True, text=True, shell=True
        )
        
        json_match = re.search(r'\{.*\}', result.stdout, re.DOTALL)
        if json_match:
            mappings = json.loads(json_match.group())
            all_mappings.extend(mappings['mappings'])
        else:
            print(f'    Warning: Failed to parse batch')
        
        time.sleep(1)  # rate limit
    
    print(f'  ✓ Mapped {len(all_mappings)} titles')
    
    # Group by O*NET tag
    grouped = defaultdict(list)
    for m in all_mappings:
        grouped[m['onet']].append(m['raw'])
    
    # Create output DataFrame
    output_data = []
    for onet_tag, titles in grouped.items():
        output_data.append({
            'onet_tag': onet_tag,
            'company_name': company_name,
            'raw_titles': ';'.join(titles)
        })
    
    return pd.DataFrame(output_data)

In [ ]:
# Process all top 10 companies
all_results = []

for idx, row in top_companies.iterrows():
    company_name = row['company_name']
    
    # Skip if already processed
    output_file = os.path.join(output_dir, f"{company_name.replace('/', '_')}_tagged.csv")
    if os.path.exists(output_file):
        print(f'\nSkipping {company_name} (already processed)')
        df_existing = pd.read_csv(output_file)
        all_results.append(df_existing)
        continue
    
    # Process company
    df_result = map_titles_for_company(row, onet_titles)
    
    # Save individual result with BOM for Power BI compatibility
    df_result.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f'  ✓ Saved to {output_file}')
    
    all_results.append(df_result)
    
    # Wait between companies
    if idx < len(top_companies) - 1:
        time.sleep(2)

print('\n=== All companies processed ===')

In [ ]:
# Combine all results
df_combined = pd.concat(all_results, ignore_index=True)

combined_output = os.path.join(output_dir, 'all_companies_tagged.csv')
df_combined.to_csv(combined_output, index=False, encoding='utf-8-sig')

print(f'\n✓ Combined results saved to {combined_output}')
print(f'\nStats:')
print(f'  Total companies: {df_combined["company_name"].nunique()}')
print(f'  Total O*NET mappings: {len(df_combined)}')
print(f'  Unique O*NET tags used: {df_combined["onet_tag"].nunique()}')

In [8]:
# Summary by company
print('\nSummary by company:')
summary = df_combined.groupby('company_name').agg(
    onet_categories=('onet_tag', 'nunique')
).reset_index()

summary = summary.merge(
    top_companies[['company_name', 'unique_titles_count']], 
    on='company_name'
)

print(summary.to_string(index=False))


Summary by company:
                                  company_name  onet_categories  unique_titles_count
                                        Amazon               80                  286
                                  Apex Systems               99                  287
                                          Dice               74                  385
                                Insight Global              139                  359
Liberty Healthcare and Rehabilitation Services               16                  265
                                        Macy's               34                  306
                     Maxim Healthcare Staffing               55                  247
                                    TEKsystems               98                  395
                               The Job Network              194                  724
                                VolunteerMatch               85                  292


In [9]:
# Preview top O*NET categories
print('\nTop 10 most common O*NET categories across all companies:')
onet_counts = df_combined['onet_tag'].value_counts().head(10)
for i, (tag, count) in enumerate(onet_counts.items(), 1):
    print(f'{i:2d}. {tag:<50s} ({count} companies)')


Top 10 most common O*NET categories across all companies:
 1. Customer Service Representatives                   (8 companies)
 2. Software Developers                                (8 companies)
 3. Market Research Analysts and Marketing Specialists (8 companies)
 4. Computer and Information Systems Managers          (8 companies)
 5. General and Operations Managers                    (7 companies)
 6. Project Management Specialists                     (7 companies)
 7. Marketing Managers                                 (7 companies)
 8. Computer Systems Engineers/Architects              (7 companies)
 9. Human Resources Specialists                        (7 companies)
10. Medical and Health Services Managers               (7 companies)


In [10]:
# Cleanup temp file
if os.path.exists('temp_prompt.txt'):
    os.remove('temp_prompt.txt')
    print('\n✓ Cleaned up temp_prompt.txt')


✓ Cleaned up temp_prompt.txt
